# 03 · Reconstrucción densa, malla y escala métrica

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StevenFAP1815/project-reconstruction-3d/blob/main/notebooks/03_dense_mesh_scale.ipynb)

A partir del modelo disperso de `02` (criterio 6 de la rúbrica):

1. **MVS (estéreo multivista):** corrección de distorsión, PatchMatch Stereo
   (profundidad por píxel) y fusión en una nube densa.
2. **Limpieza:** recorte alrededor del objeto, eliminación de la mesa (RANSAC)
   y de las islas de ruido (componente conexa más grande).
3. **Malla:** reconstrucción de superficie de Poisson y recorte de la
   superficie inventada.
4. **Escala métrica:** el modelo se escala con una medida física del objeto y
   el error dimensional se calcula con las demás (estrategia A,
   `docs/capture-protocol.md`).
5. **Exportación:** nube y malla en milímetros (`.ply`, `.obj`) y `.glb` (en
   metros, como exige glTF) para visores web.

El objeto se elige con `OBJECT`. **Entorno:** GPU (PatchMatch solo funciona con CUDA).

## 1. Entorno

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/StevenFAP1815/project-reconstruction-3d.git"
REPO_DIR = Path("/content/project-reconstruction-3d")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

# Python importa cada módulo una sola vez por sesión. Si esta sesión ya había
# cargado recon3d (una ejecución anterior), se descarta para que el siguiente
# import lea el código recién descargado y no una versión antigua en memoria.
for module in [name for name in sys.modules if name == "recon3d" or name.startswith("recon3d.")]:
    del sys.modules[module]

COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Código del repositorio en el commit", COMMIT)

In [ ]:
%pip install -q -r /content/project-reconstruction-3d/requirements-colab.txt

## 2. Parámetros

In [ ]:
OBJECT = "caja-carton-leche"   # caja-carton-leche | canasto-flores | audifonos | colador-cocina | vaso-vidrio
# Modelo disperso de partida: el de la caja con la configuración D_calibrada
# (100 % registradas); el resto, la línea base de 02.
SPARSE_METHOD = "colmap-v2" if OBJECT == "caja-carton-leche" else "colmap"
METHOD = "colmap-mvs"
SEED = 42
MIN_REGISTERED = 10            # con menos fotos registradas no se intenta MVS

MAX_IMAGE_SIZE = 1600          # lado máximo (px) con el que trabaja MVS
PATCH_MATCH_OPTIONS = {"window_radius": 5, "num_iterations": 5}
FUSION_OPTIONS = {"min_num_pixels": 5}

# Limpieza: distancias relativas a la distancia mediana cámara-objeto.
CROP_RATIO = 0.6               # radio del recorte alrededor del objeto
PLANE_RATIO = 0.01             # tolerancia del plano de la mesa
VOXEL_RATIO = 0.01             # celda para la componente conexa
MIN_POINTS_PER_VOXEL = 3       # celdas con menos puntos = ruido

POISSON_OPTIONS = {"depth": 10, "trim": 7}

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import pycolmap

from recon3d.dataset import extract_object_zip
from recon3d.paths import get_paths
from recon3d.reproducibility import set_seed
from recon3d.results import environment_info, write_meta

set_seed(SEED)
paths = get_paths()
paths.create()
start_total = time.perf_counter()

LOCAL_IMAGES = Path("/content/data") / OBJECT
images = extract_object_zip(paths.data_zip(OBJECT), LOCAL_IMAGES)
sparse_dir = paths.run_dir(OBJECT, SPARSE_METHOD) / "sparse"
rec = pycolmap.Reconstruction(str(sparse_dir))
run_dir = paths.run_dir(OBJECT, METHOD)
run_dir.mkdir(parents=True, exist_ok=True)

print(f"{OBJECT}: {len(images)} fotos | modelo disperso {SPARSE_METHOD}: "
      f"{rec.num_reg_images()} registradas, {rec.num_points3D()} puntos 3D")
print("pycolmap", pycolmap.__version__, "| CUDA:", pycolmap.has_cuda)
print("GPU:", environment_info()["gpu"])

if rec.num_reg_images() < MIN_REGISTERED:
    write_meta(run_dir, object_name=OBJECT, method=METHOD, n_images=len(images),
               n_registered=rec.num_reg_images(), runtime_s=time.perf_counter() - start_total,
               seed=SEED, extra={"status": "omitido", "sparse_method": SPARSE_METHOD,
                                 "reason": f"menos de {MIN_REGISTERED} fotos registradas en SfM"})
    raise RuntimeError(f"{OBJECT}: solo {rec.num_reg_images()} fotos registradas; no hay vistas "
                     "suficientes para MVS. Se registra como fallo del método clásico.")

## 3. Reconstrucción densa (MVS)

| Etapa | Qué hace |
|---|---|
| Corrección de distorsión | Reproyecta cada foto a una cámara pinhole ideal, sin distorsión. |
| PatchMatch Stereo | Estima profundidad y normal en cada píxel comparando ventanas de `2·window_radius+1` px con las fotos vecinas; con consistencia geométrica, las profundidades de fotos distintas deben coincidir. |
| Fusión | Une los mapas de profundidad en una nube; un punto se acepta si lo confirman al menos `min_num_pixels` fotos. |

`MAX_IMAGE_SIZE` limita la resolución: a 1600 px la caja tarda unos 15-25 min
en una T4; a resolución completa (4000 px) tardaría varias veces más sin
mejorar mucho la medida del objeto.

In [ ]:
from recon3d.dense import run_mvs

WORK = Path("/content/work") / OBJECT / METHOD
mvs_report = {}
fused_path, mvs_timings = run_mvs(
    sparse_dir, LOCAL_IMAGES, WORK,
    max_image_size=MAX_IMAGE_SIZE,
    patch_match_options=PATCH_MATCH_OPTIONS,
    fusion_options=FUSION_OPTIONS,
    report=mvs_report,
)
if mvs_report["ignored_options"]:
    print("Opciones no reconocidas:", mvs_report["ignored_options"])
for key, value in mvs_timings.items():
    print(f"{key:>15}: {value:8.1f} s")

In [ ]:
import cv2
import matplotlib.pyplot as plt

from recon3d.dense import read_colmap_array

depth_dir = WORK / "stereo" / "depth_maps"
depth_files = sorted(depth_dir.glob("*.geometric.bin"))
example = depth_files[len(depth_files) // 2]
name = example.name.replace(".geometric.bin", "")
depth = read_colmap_array(example)
photo = cv2.cvtColor(cv2.imread(str(WORK / "images" / name)), cv2.COLOR_BGR2RGB)

valid = depth > 0
fig, axes = plt.subplots(1, 2, figsize=(11, 7))
axes[0].imshow(photo)
axes[0].set_title(f"Foto sin distorsión: {name}")
shown = axes[1].imshow(np.where(valid, depth, np.nan), cmap="turbo",
                       vmin=np.percentile(depth[valid], 2), vmax=np.percentile(depth[valid], 98))
axes[1].set_title(f"Mapa de profundidad geométrico ({100 * valid.mean():.0f} % de píxeles con valor)")
fig.colorbar(shown, ax=axes[1], label="profundidad (unidades SfM)", shrink=0.7)
for ax in axes:
    ax.axis("off")
plt.show()

## 4. Limpieza de la nube densa

La nube fusionada contiene, además del objeto, la mesa y el fondo. Se aísla el
objeto en tres pasos (`recon3d.pointcloud.clean_object_cloud`):

1. **Recorte esférico** alrededor del punto al que miran todas las cámaras.
2. **Plano de la mesa** con RANSAC: se acepta solo si es casi horizontal y queda
   por debajo de casi todo; se eliminan sus puntos y lo que está debajo.
3. **Componente conexa más grande**: el objeto es la mayor isla de puntos.

La vertical aproximada se obtiene de las órbitas de la cámara (PCA de los
centros, como en la trayectoria de `02`).

In [ ]:
from recon3d.dense import viewing_directions
from recon3d.geometry import trajectory_frame
from recon3d.pointcloud import clean_object_cloud, look_at_point, read_ply

cloud = read_ply(fused_path)
points = cloud["points"]
centers, directions = viewing_directions(rec)
_, orbit_axes = trajectory_frame(centers)
up = orbit_axes[2]
if np.mean((centers - look_at_point(centers, directions)) @ up) < 0:
    up = -up  # la vertical apunta hacia las cámaras, que están por encima del objeto

mask, clean_report = clean_object_cloud(
    points, centers, directions, up,
    crop_ratio=CROP_RATIO, plane_ratio=PLANE_RATIO, voxel_ratio=VOXEL_RATIO,
    min_points_per_voxel=MIN_POINTS_PER_VOXEL, seed=SEED,
)
stages = pd.DataFrame({
    "etapa": ["nube fusionada", "recorte esférico", "sin mesa", "componente principal"],
    "puntos": [clean_report[k] for k in ("input", "after_crop", "after_table", "after_component")],
})
stages["% del total"] = (100 * stages["puntos"] / stages["puntos"].iloc[0]).round(1)
print("Mesa detectada:", clean_report["table_plane_found"])
display(stages)

In [ ]:
def show_views(xyz, rgb, title, max_points=60000):
    rng = np.random.default_rng(SEED)
    keep = rng.choice(len(xyz), size=min(max_points, len(xyz)), replace=False)
    xyz, rgb = xyz[keep], rgb[keep] / 255
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, (i, j), label in zip(axes, [(0, 1), (0, 2), (1, 2)], ["x-y", "x-z", "y-z"]):
        ax.scatter(xyz[:, i], xyz[:, j], c=rgb, s=0.3)
        ax.set_aspect("equal")
        ax.set_title(label)
    fig.suptitle(title)
    plt.show()

# Vistas en el sistema de las órbitas (z = vertical aproximada).
center = np.array(clean_report["center"])
to_orbit = lambda p: (p - center) @ np.vstack([orbit_axes[0], np.cross(up, orbit_axes[0]), up]).T
show_views(to_orbit(points), cloud["colors"], f"Nube fusionada: {len(points):,} puntos")
show_views(to_orbit(points[mask]), cloud["colors"][mask], f"Objeto limpio: {mask.sum():,} puntos")

## 5. Escala métrica y error dimensional

El modelo de SfM no tiene unidades. Se calcula la caja envolvente de volumen
mínimo del objeto (con la vertical cerca de la de las órbitas) y se mide cada
dimensión de la tabla de `recon3d.measurements`. La medida marcada como
`escala` fija el factor mm/unidad; las demás se comparan con la medida física
(referencia de la rúbrica: error ≤ 2 %). Cada extremo se recorta un 1 % para
que los puntos sueltos no alarguen las medidas.

In [ ]:
from recon3d.measurements import MEASUREMENTS, NOT_AUTOMATIC, SCALE_BY
from recon3d.pointcloud import angular_coverage, dimensional_errors, object_frame

object_points = points[mask]
coords, rotation, origin = object_frame(object_points, clean_report["up"], seed=SEED)
scale, rows = dimensional_errors(coords, MEASUREMENTS[OBJECT], SCALE_BY[OBJECT])
measurements_df = pd.DataFrame(rows)
# Completitud: % de los 36 sectores de 10° alrededor del objeto con puntos,
# medidos alrededor del punto al que miran las cámaras.
center_xy = ((np.array(clean_report["center"]) - origin) @ rotation.T)[:2]
coverage = angular_coverage(coords, center_xy)
print(f"Factor de escala: {scale:.3f} mm por unidad de SfM (medida: {SCALE_BY[OBJECT]})")
print(f"Cobertura angular: {coverage} % de la vuelta alrededor del objeto")
if OBJECT in NOT_AUTOMATIC:
    print("No evaluadas automáticamente:", NOT_AUTOMATIC[OBJECT])
evaluation = measurements_df[measurements_df["role"] == "evaluación"]
print(f"Error absoluto medio de las medidas de evaluación: {evaluation['error_pct'].abs().mean():.2f} %")
display(measurements_df)

In [ ]:
coords_mm = coords * scale
low, high = np.percentile(coords_mm, 1, axis=0), np.percentile(coords_mm, 99, axis=0)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
rng = np.random.default_rng(SEED)
keep = rng.choice(len(coords_mm), size=min(60000, len(coords_mm)), replace=False)
for ax, (i, j), label in zip(axes, [(0, 1), (0, 2), (1, 2)], ["x-y (planta)", "x-z (alzado)", "y-z (perfil)"]):
    ax.scatter(coords_mm[keep, i], coords_mm[keep, j], c=cloud["colors"][mask][keep] / 255, s=0.3)
    ax.add_patch(plt.Rectangle((low[i], low[j]), high[i] - low[i], high[j] - low[j],
                               fill=False, color="red", linewidth=1.5))
    ax.set_xlabel("mm")
    ax.set_ylabel("mm")
    ax.set_aspect("equal")
    ax.set_title(label)
fig.suptitle(f"{OBJECT} escalado: caja envolvente {np.round(high - low, 1)} mm")
plt.show()

## 6. Malla de Poisson

La reconstrucción de Poisson busca una superficie cerrada cuyas normales
coincidan con las de los puntos. `depth` fija la resolución del octree (10 ≈
1/1024 del tamaño de la escena) y `trim` elimina las zonas con muy pocos
puntos de apoyo. Después se recortan los triángulos fuera de la caja del objeto
(Poisson también cierra las zonas no vistas, como la base apoyada en la mesa).

In [ ]:
from recon3d.dense import poisson_mesh
from recon3d.pointcloud import crop_mesh, write_ply

clean_path = WORK / "object_clean.ply"
write_ply(clean_path, object_points, colors=cloud["colors"][mask], normals=cloud["normals"][mask])
mesh_report = {}
start = time.perf_counter()
mesh_raw = poisson_mesh(clean_path, WORK / "mesh_poisson.ply", options=POISSON_OPTIONS, report=mesh_report)
poisson_s = time.perf_counter() - start
if mesh_report["ignored_options"]:
    print("Opciones no reconocidas:", mesh_report["ignored_options"])

mesh = read_ply(mesh_raw)
vertices_mm = (mesh["points"] - origin) @ rotation.T * scale
margin = 0.03 * (high - low)
vertices_mm, faces, used = crop_mesh(vertices_mm, mesh["faces"], low - margin, high + margin)
mesh_colors = mesh["colors"][used] if "colors" in mesh else None
print(f"Poisson: {len(mesh['points']):,} vértices y {len(mesh['faces']):,} triángulos en {poisson_s:.1f} s")
print(f"Tras el recorte: {len(vertices_mm):,} vértices y {len(faces):,} triángulos")

In [ ]:
import plotly.graph_objects as go

# Para que el notebook no pese decenas de MB se muestra una muestra de triángulos
# y solo los vértices que usan.
rng = np.random.default_rng(SEED)
shown_faces = faces[rng.choice(len(faces), size=min(60000, len(faces)), replace=False)]
shown_vertices, shown_faces = np.unique(shown_faces, return_inverse=True)
shown_faces = shown_faces.reshape(-1, 3)
xyz = vertices_mm[shown_vertices]
color = (None if mesh_colors is None
         else [f"rgb({r},{g},{b})" for r, g, b in mesh_colors[shown_vertices]])
fig = go.Figure(go.Mesh3d(
    x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
    i=shown_faces[:, 0], j=shown_faces[:, 1], k=shown_faces[:, 2],
    vertexcolor=color, color=None if color else "lightgray", flatshading=True,
))
fig.update_layout(height=650, scene=dict(aspectmode="data", xaxis_title="mm",
                                         yaxis_title="mm", zaxis_title="mm"),
                  title=f"Malla de {OBJECT} en mm (muestra de {len(shown_faces):,} triángulos)")
fig.show()

## 7. Guardar resultados

En `results/<objeto>/colmap-mvs/` de Drive:

- `fused.ply`: nube densa completa, sin limpiar (unidades de SfM).
- `object_clean_mm.ply`: nube del objeto limpia y escalada (mm).
- `mesh_mm.ply` y `mesh_mm.obj`: malla escalada (mm).
- `mesh.glb`: malla para visores web (glTF mide en metros).
- `measurements.csv`, `cleaning.csv` y `meta.json`.

In [ ]:
import shutil

from recon3d.pointcloud import write_glb, write_obj

shutil.copy(fused_path, run_dir / "fused.ply")
write_ply(run_dir / "object_clean_mm.ply", coords_mm, colors=cloud["colors"][mask],
          normals=cloud["normals"][mask] @ rotation.T)
write_ply(run_dir / "mesh_mm.ply", vertices_mm, colors=mesh_colors, faces=faces)
write_obj(run_dir / "mesh_mm.obj", vertices_mm, faces, mesh_colors)
write_glb(run_dir / "mesh.glb", vertices_mm / 1000.0, faces, mesh_colors)
measurements_df.to_csv(run_dir / "measurements.csv", index=False)
stages.to_csv(run_dir / "cleaning.csv", index=False)

meta_path = write_meta(
    run_dir,
    object_name=OBJECT,
    method=METHOD,
    n_images=len(images),
    n_registered=rec.num_reg_images(),
    runtime_s=time.perf_counter() - start_total,
    seed=SEED,
    scale_factor=float(scale),
    extra={
        "status": "ok",
        "sparse_method": SPARSE_METHOD,
        "max_image_size": MAX_IMAGE_SIZE,
        "patch_match_options": PATCH_MATCH_OPTIONS,
        "fusion_options": FUSION_OPTIONS,
        "poisson_options": POISSON_OPTIONS,
        "cleaning": {"crop_ratio": CROP_RATIO, "plane_ratio": PLANE_RATIO,
                     "voxel_ratio": VOXEL_RATIO, "min_points_per_voxel": MIN_POINTS_PER_VOXEL,
                     **{k: v for k, v in clean_report.items() if k not in ("center", "up")}},
        "timings_s": {**{k: round(v, 1) for k, v in mvs_timings.items()}, "poisson_s": round(poisson_s, 1)},
        "n_fused_points": int(len(points)),
        "n_object_points": int(mask.sum()),
        "n_mesh_vertices": int(len(vertices_mm)),
        "n_mesh_faces": int(len(faces)),
        "scale_by": SCALE_BY[OBJECT],
        "bounding_box_mm": np.round(high - low, 2).tolist(),
        "measurements": rows,
        "mean_abs_error_pct": round(float(evaluation["error_pct"].abs().mean()), 2),
        "angular_coverage_pct": coverage,
    },
)
for path in sorted(run_dir.iterdir()):
    print(f"{path.stat().st_size / 1e6:8.2f} MB  {path.name}")

## Siguiente paso

- Descargar el `.ipynb` ejecutado como `03_dense_<objeto>.ipynb` en
  `notebooks/runs/` y `results/<objeto>/colmap-mvs` como zip en
  `results/<objeto>/`.
- Repetir con los demás objetos cambiando `OBJECT`. Los objetos con menos de
  `MIN_REGISTERED` fotos registradas se detienen en la sección 2 y quedan
  registrados como fallo del método clásico.
- Abrir `mesh.glb` en un visor web (por ejemplo, https://gltf-viewer.donmccurdy.com).